# DLT pipeline or LakeFlow Pipeline

## Streaming table

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import *
from pyspark.sql.types import *

#### Expectation

In [0]:
gold_products_rules = {
    "rule1" : "product_id IS NOT NULL",
    "rule2" : "product_name IS NOT NULL"
}

In [0]:
@dp.table()
@dp.expect_all_or_drop(gold_products_rules)
def DimProducts_Stage():
    df = spark.readStream.table("databricks_pipeline_cata.silver.products")
    return df


#### Streaming Views

In [0]:
@dp.temporary_view()
def DimProducts_View():
    df = spark.readStream.table("Live.DimProducts_Stage")
    return df

#### DimProducts

In [0]:
dp.create_streaming_table("DimProducts")

In [0]:
dp.create_auto_cdc_flow(
  target = "DimProducts",
  source = "Live.DimProducts_View",
  keys = ["product_id"],
  sequence_by = "product_id",
  stored_as_scd_type = "2"
)